# Lecture 4 - NumPy

**Lecture Topic**: Scripts vs notebooks and regular expressions, then NumPy arrays, vectorization and broadcasting, and a first look at what a Python loop costs

**Instructor**: Duncan Watson-Parris

## Announcements

- Project teams and the menu of project briefs are released today. Your brief choice (or your own proposal) is due next Friday, Oct 9.
- The first weekly quiz is due on Sunday at 11:59pm.
- ClassBuzz poll for today: `M6U2`

## Recap

In the last lecture, we covered the following topics:
 - Functions
 - Comprehensions, lambdas and generators
 - Exceptions, and how to read a traceback
 - Files and modules

Today, we will cover the following topics:
 - Scripts vs notebooks
 - Regular expressions, and how to check them
 - NumPy arrays and dtypes
 - Indexing, masks and multidimensional arrays
 - Operations and broadcasting
 - A first look at cost: Python loops vs NumPy

## Scripts vs notebooks

We didn't quite get to this on Wednesday, so let's finish it off before moving on to regular expressions.

Notebooks are fantastic for learning and teaching, as they allow you to write code and see the output immediately.
For more intensive analysis code, you may want to use a Python script, which can be run from the command line.

Any `.py` file is a module you can import, and also a script you can run:

In [ ]:
%%writefile fares.py
def parse_fare(row):
    """Return the fare of a trip as a float."""
    return float(row["fare"])

if __name__ == "__main__":
    print(parse_fare({"fare": "12.5"}))

In [ ]:
!python fares.py

In [ ]:
import fares
fares.parse_fare({"fare": "8.0"})

The code under `if __name__ == "__main__":` only runs when the file is executed directly, not when it is imported.

### A word of warning about notebooks

Cells can be run in any order, and the notebook remembers everything you've run, even cells you've since deleted.

In [ ]:
threshold = 10

In [ ]:
trip_fares = [5, 12, 30]
[f for f in trip_fares if f > threshold]

Before trusting a notebook (yours or anyone else's), do **Kernel ▸ Restart & Run All** and check that you get the same answer.

## Regular Expressions

String manipulation is a common problem when preparing data for analysis.

We've already seen the `split()` and `replace()` methods, among others. But what if we want to find more complex patterns?

One example might be searching for phone numbers (e.g. 999-999-9999) in a text database.

Regular expressions are text matching patterns described with a formal syntax. You'll often hear regular expressions referred to as 'regex' or 'regexp' in conversation.

Last year I finished the regex lecture by saying *"ChatGPT is also pretty good at this...!"*. That is even more true now, so this year we'll spend less time writing regular expressions and more time checking them.

### Searching for patterns in text

In [ ]:
import re

text = "Call 858-534-2230 or 619-555-0199 for details."

print(re.search(r"\d{3}-\d{3}-\d{4}", text))    # the first match (or None)
print(re.findall(r"\d{3}-\d{3}-\d{4}", text))   # all the matches

### Pattern syntax

| Pattern | Matches |
|---|---|
| `\d`, `\w`, `\s` | a digit, an alphanumeric character, whitespace |
| `.` | any character except a newline |
| `[abc]`, `[a-z]` | any one of a, b or c; any lowercase letter |
| `[^abc]` | any character *except* a, b or c |
| `*`, `+`, `?` | zero or more, one or more, zero or one of the previous item |
| `{m}`, `{m,n}` | exactly m, or between m and n, of the previous item |
| `^`, `$` | the start and end of the string |

Use raw strings (`r"..."`) for regular expressions, otherwise every backslash has to be escaped.

### Quick quiz

How many matches will this find?

 - A. 0
 - B. 5
 - C. 7
 - D. I don't know

In [ ]:
test_phrase = 'sdsd..sssddd...sdddsddd...dsds...dsssss...sdddd'

print(re.findall('sd+',  # s followed by one or more d's
                 test_phrase))

## Checking a regular expression

Suppose I ask an agent for a function that checks whether a string is a valid US phone number, in the form 999-999-9999. It gives me:

In [ ]:
def is_phone_number(s):
    return re.search(r"\d{3}-\d{3}-\d{4}", s) is not None

is_phone_number("858-534-2230")

### Quick quiz

Is this function correct?

The way to check is to write down some examples where you know the answer, including ones that should **fail**, and test them all:

In [ ]:
examples = {
    "858-534-2230": True,
    "858-534-223": False,
    "858 534 2230": False,
    "8585-534-22301": False,
    "Phone: 858-534-2230": False,
}

def check(func):
    for s, expected in examples.items():
        result = func(s)
        flag = "" if result == expected else "  <-- WRONG"
        print(f"{s!r:24} expected {expected!s:5}  got {result!s:5}{flag}")

check(is_phone_number)

`re.search` looks for the pattern *anywhere* in the string. We want the whole string to match, which is what `re.fullmatch` does:

In [ ]:
def is_phone_number(s):
    return re.fullmatch(r"\d{3}-\d{3}-\d{4}", s) is not None

check(is_phone_number)

The pattern itself was fine. The bug was `search` versus `fullmatch`, which is very easy to miss when reading.

A couple of other things worth noting:

- Whether `"858 534 2230"` should count as valid is a question for whoever asked for the function. That's a specification problem, not a regex problem.
- A list of inputs with the answers you expect is exactly what a unit test is. We'll come back to this in Module D.

For more patterns, take a look at the full [documentation](https://docs.python.org/3/library/re.html), and [regex101.com](https://regex101.com) is very handy for seeing what a pattern matches.

## NumPy

NumPy provides an efficient representation of multidimensional datasets like vectors and matrices, and tools for linear algebra and general matrix manipulations - essential building blocks of virtually all technical computing.

pandas and xarray are both built on top of NumPy, so everything today carries straight over to the next three lectures.

Typically NumPy is imported as `np`:

In [ ]:
import numpy as np

NumPy, at its core, provides a powerful array object. Let's start by exploring how the NumPy array differs from a Python list:

In [ ]:
lst = [10, 20, 30, 40]
arr = np.array([10, 20, 30, 40])
print(lst, arr)
print(lst[0], arr[0])
print(lst[2:], arr[2:])

### Differences between arrays and lists

The first difference to note between lists and arrays is that arrays are *homogeneous*; i.e. all elements of an array must be of the same type. In contrast, lists can contain elements of arbitrary type.

In [ ]:
lst[-1] = 'a string inside a list'
lst

In [ ]:
arr[-1] = 'a string inside an array'

### Data types

The information about the type of an array is contained in its `dtype` attribute.
Once an array has been created, its `dtype` is fixed (in this case to an 8 byte/64 bit signed integer) and it can only store elements of the same type.

In [ ]:
arr.dtype

If we try storing a floating point number in this array it will be automatically converted into an integer:

In [ ]:
arr[-1] = 1.234
arr

Note that this happens silently. There is no error and no warning.

### Pop quiz

What is the output of the below code?

 - A. `[300 200]`
 - B. `[ 44 200]`
 - C. `[255 200]`
 - D. An error

In [ ]:
a = np.array([200, 100], dtype=np.uint8)
b = np.array([100, 100], dtype=np.uint8)
a + b

A `uint8` can only hold 0 to 255, so 300 wraps around to 300 - 256 = 44. Again, no error and no warning.

This matters any time data is stored in a small type to save space, like images (usually `uint8`) or some sensor and climate data.

## Array Properties and Methods

The following provide basic information about the size, shape and data in the array:

In [ ]:
arr = np.array([10, 20, 30, 40])
print('Data type                :', arr.dtype)
print('Total number of elements :', arr.size)
print('Number of dimensions     :', arr.ndim)
print('Shape (dimensionality)   :', arr.shape)
print('Memory used (in bytes)   :', arr.nbytes)

Arrays also have many useful statistical/mathematical methods:

In [ ]:
print('Minimum and maximum             :', arr.min(), arr.max())
print('Sum and product of all elements :', arr.sum(), arr.prod())
print('Mean and standard deviation     :', arr.mean(), arr.std())

## Creating Arrays

In [ ]:
print(np.zeros(5))
print(np.ones(3, dtype=np.int32))
print(np.arange(5, 7, 0.5))     # like range, but returns an array
print(np.linspace(0, 1, 5))     # 5 points, including both ends

It is often useful to create arrays with random numbers that follow a specific distribution:

In [ ]:
rng = np.random.default_rng(seed=42)
norm10 = rng.normal(10, 3, 5)   # mean 10, standard deviation 3
norm10

Setting the seed means you get the same "random" numbers every time, which makes your results reproducible.

## Indexing with other arrays

Arrays also allow for a more sophisticated kind of indexing that is very powerful: you can index an array with another array, and in particular with an array of boolean values.

This is particularly useful to extract information from an array that matches a certain condition.

In [ ]:
mask = norm10 > 9
mask

In [ ]:
print('Values above 9:', norm10[mask])
norm10[mask] = 0
print('After resetting them to 0:', norm10)

### Quick quiz

How many elements of `nanarray` are `> 0.5`, and how many are `<= 0.5`?

In [ ]:
nanarray = np.array([-1., 0., 1., np.nan])

In [ ]:
print(nanarray > 0.5)
print(nanarray <= 0.5)

`NaN` compares as `False` to everything, so it ends up in neither group. If you split data with a mask like this, the missing values quietly drop out of both halves.

Similarly, `np.mean` of an array with a `NaN` in it is `NaN`. `np.nanmean` ignores them, but you should know how many you're ignoring.

## Arrays with more than one dimension

NumPy can also create arrays of arbitrary dimensions, and all the methods above work on arrays with more than one dimension.

In [ ]:
lst2 = [[1, 2, 3], [4, 5, 6]]
arr2 = np.array([[1, 2, 3], [4, 5, 6]])
print(arr2)
print(arr2.shape)

While nested lists can be indexed by repeatedly using the `[ ]` operator, multidimensional arrays support a much more natural indexing syntax using a single `[ ]` and a set of indices separated by commas:

In [ ]:
print(lst2[0][1])
print(arr2[0, 1])

Question: Why does the following example produce different results?

In [ ]:
print(lst2[0:2][1])
print(arr2[:, 1])

### Reshaping and slicing

The shape of an array can be changed at any time, as long as the total number of elements is unchanged. `-1` means "work this one out for me":

In [ ]:
arr = np.arange(8).reshape(-1, 4)
print(arr)
print(arr[1, 2:])   # second row, last two columns

`...` (ellipsis) expands to `:` for all of the dimensions you haven't specified.

### Quick quiz

What is the shape of this slice?

 - A. `(3, 6, 7, 4)`
 - B. `(1, 6, 7, 4)`
 - C. `(6, 7, 5)`
 - D. `(6, 7, 4)`

In [ ]:
arr3 = np.empty((5, 6, 7, 8))
print(arr3[2:3, ..., :4].shape)

## Operating with arrays

It is important to remember that in general, all operations with arrays are applied *element-wise*, i.e., are applied to all the elements of the array at the same time.

In [ ]:
arr1 = np.arange(4)
arr2 = np.arange(10, 14)
print(arr1, '+', arr2, '=', arr1 + arr2)
print(arr1, '*', arr2, '=', arr1 * arr2)

Importantly, even the multiplication operator is by default applied element-wise: It is *not* the matrix multiplication from linear algebra. For that, use `@` (or `np.dot`).

NumPy also ships with a full complement of mathematical functions that work on entire arrays (`np.sin`, `np.exp`, `np.log`, ...).

For multidimensional arrays it is possible to carry out computations along a single dimension by passing the `axis` parameter:

In [ ]:
arr = np.arange(6).reshape((2, 3))
print('For the following array:\n', arr)
print('The sum of elements along the rows is    :', arr.sum(axis=1))
print('The sum of elements along the columns is :', arr.sum(axis=0))

The value of the `axis` parameter is the dimension that will be *consumed* once the operation has been carried out. This is why to sum along the columns we use `axis=0`.

## Broadcasting

The fact that NumPy operates on an element-wise basis means that in principle arrays must always match one another's shape. However, NumPy will also helpfully "*broadcast*" dimensions when possible.

In [ ]:
print(np.arange(3) + 5)                 # a scalar to a 1D array
print(np.ones((3, 3)) + np.arange(3))   # a 1D array to every row of a 2D array

We can also broadcast in two directions at a time:

In [ ]:
a = np.arange(3).reshape((3, 1))
b = np.arange(3)
print(a, '+', b, '=\n', a + b)

Pictorially:

![Illustration of broadcasting](images/fig_broadcast_visual_1.png)

([image source](http://www.astroml.org/book_figures/appendix/fig_broadcast_visual.html))

### Rules of Broadcasting

When operating on two arrays, NumPy compares their shapes element-wise. It starts with the trailing dimensions, and works its way forward. Two dimensions are compatible when

* they are equal, or
* one of them is 1 (or missing); in this case NumPy will 'stretch' it to match the other.

If these conditions are not met, a `ValueError: operands could not be broadcast together` exception is thrown.

Note that all of this happens without ever actually creating the expanded arrays in memory, which can save lots of memory when the arrays in question are large.

### Questions:

What will the result of adding `arr1` with `arr2` be in the following cases?

In [ ]:
arr1 = np.ones((2, 3))
arr2 = np.ones((2, 1))

arr1 + arr2

In [ ]:
arr1 = np.ones((1, 3))
arr2 = np.ones((2, 1))

arr1 + arr2

In [ ]:
arr1 = np.ones((1, 3))
arr2 = np.ones((1, 2))

arr1 + arr2

### Quick quiz

What will be the shape of `C` given the following arrays and operation:

```python
A = np.ones((3, 1, 5))
B = np.ones((1, 4, 1))
C = A + B
```

### Quick quiz: spot the bug

An agent was asked for a function that removes each station's average from an array of temperature readings with shape `(stations, days)`. It wrote this, and tested it:

In [ ]:
def remove_station_mean(data):
    return data - data.mean(axis=1)

test = np.array([[1., 2., 3.],
                 [4., 5., 6.],
                 [7., 8., 9.]])
remove_station_mean(test)

Is it correct?

If it were, every row would now have a mean of zero:

In [ ]:
remove_station_mean(test).mean(axis=1)

`data.mean(axis=1)` has shape `(3,)`, which broadcasts against the *last* dimension, so each **column** had a different station's mean taken off it.

And on real data, with 4 stations and 7 days:

In [ ]:
remove_station_mean(np.ones((4, 7)))

The fix is to keep the dimension we averaged over, so it broadcasts the right way:

In [ ]:
def remove_station_mean(data):
    return data - data.mean(axis=1, keepdims=True)

print(remove_station_mean(test).mean(axis=1))
print(remove_station_mean(np.ones((4, 7))).shape)

The lesson: **test with arrays that have a different length in every dimension**. A square array can't tell rows from columns.

## Views, not Copies

Note that reshaping and slicing (like most NumPy operations), wherever possible, provide a **view** of *the same memory*:

In [ ]:
arr = np.arange(8)
arr_view = arr.reshape(2, 4)
arr_view[0, 0] = 1000
print(arr)

This lack of copying allows for very efficient vectorized operations, but this power should be used carefully - if used badly it can lead to some bugs that are hard to track down.

If in doubt, you can always copy the data to a different block of memory with the **`copy()`** method.

## A first look at cost

Let's compute the tip as a percentage of the fare for a million taxi trips, first with a Python loop, then with NumPy.

In [ ]:
rng = np.random.default_rng(seed=0)
fares = rng.uniform(5, 60, size=1_000_000)
tips = rng.uniform(0, 10, size=1_000_000)

def tip_pct_loop(fares, tips):
    out = []
    for i in range(len(fares)):
        out.append(tips[i] / fares[i] * 100)
    return out

def tip_pct_numpy(fares, tips):
    return tips / fares * 100

### Quick quiz

How much faster do you think the NumPy version will be?

 - A. About the same
 - B. 2-5x faster
 - C. 10-100x faster
 - D. More than 1000x faster

In [ ]:
fares_list, tips_list = fares.tolist(), tips.tolist()

%timeit tip_pct_loop(fares_list, tips_list)
%timeit tip_pct_numpy(fares, tips)

In [ ]:
np.allclose(tip_pct_loop(fares_list, tips_list), tip_pct_numpy(fares, tips))

### Why?

Each time around a Python loop, the interpreter has to look up what `tips[i]` is, check its type, work out what `/` means for that type, and create a new Python object for the result.

NumPy does that once for the whole array, then runs a tight loop in compiled C over one contiguous block of memory.

The memory is different too. A NumPy array stores just the numbers, whereas a list stores a pointer to a separate Python object for each one:

In [ ]:
import sys

print(f"array: {fares.nbytes / 1e6:.0f} MB")
print(f"list:  {(sys.getsizeof(fares_list) + sum(sys.getsizeof(x) for x in fares_list)) / 1e6:.0f} MB")

Both versions give the same answer, and on 10,000 rows you would never notice the difference. At 100 million rows you would.

This is exactly the kind of choice an agent makes for you without mentioning it. We'll come back to it properly in Module C, where we'll measure how the cost grows with the size of the data.

## Summary

Today, we covered the following topics:
 - Scripts vs notebooks
 - Regular expressions, and how to check them
 - NumPy arrays and dtypes (and what happens silently when they don't fit)
 - Indexing, boolean masks and `NaN`
 - Multidimensional arrays, `axis` and broadcasting
 - Views vs copies
 - Python loops vs NumPy: time and memory

On Monday, pandas.